#  PDS - Project - Analysis of Statistical Similarity of Network Services
**Author**: Matúš Remeň (xremen01)\
**Academic year**: 2023/24

---
# Statistical Analysis of Network Flows

## Dataset
The dataset contains network flows in IPFIX format, which were captured by the open-source probe [ipfixprobe](https://github.com/CESNET/ipfixprobe).
The data is pre-processed and stored in an Apache Parquet file.
Flows are limited to **HTTPS traffic (TCP with dst port 443)** and contain the following fields ([source](https://github.com/CESNET/ipfixprobe/blob/master/README.md)):
- `string TLS_SNI` - TLS Server Name Indication field from client
- `time TIME_FIRST` - timestamp of the first packet
- `time TIME_LAST` - timestamp of the last packet
- `uint32 PACKETS` - number of packets in data flow (src -> dst)
- `uint32 PACKETS_REV` - number of packets in data flow (src <- dst)
- `uint64 BYTES` - number of bytes in data flow (src -> dst)
- `uint64 BYTES_REV` - number of bytes in data flow (src <- dst)
- `uint8 TCP_FLAGS` - TCP protocol flags (src -> dst) (all flags which were used in the flow)
- `int8* PPI_PKT_DIRECTIONS` - directions of the first PSTATS_MAXELEMCOUNT packets (1 = src -> dst, -1 = src <- dst)
- `time* PPI_PKT_TIMES` - timestamps of the first PSTATS_MAXELEMCOUNT packets
- `uint16* PPI_PKT_LENGTHS` - sizes of the first PSTATS_MAXELEMCOUNT packets
- `uint8* PPI_PKT_FLAGS` - TCP flags of the first PSTATS_MAXELEMCOUNT packets

## Load the dataset, and display basic statistical information about attributes

In [ ]:
# Import libraries.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
pd.set_option("display.float_format", "{:.8f}".format)

# Define constants.
INPUT_FILE_PATH = "tls-pds-07-03-2024.parquet"
# PSTATS_MAXELEMCOUNT = 100

In [ ]:
# Load dataset - network flows - from an Apache Parquet file.
df: pd.DataFrame = pd.read_parquet(INPUT_FILE_PATH)
print(f"{df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display the first two rows of the dataset.
df.head(2)

In [ ]:
# Count the number of unique TLS_SNI values in the dataset.
df["string TLS_SNI"].nunique()

In [ ]:
# Display statistical data about dataset columns/attributes.
df.describe()

## Additional data preprocessing, and attribute engineering
The dataset contains records of more than 1.2 million network flows, which is a significant amount of data. I decided to further
preprocess the data with an attempt to meaningfully reduce that number, while preserving the characteristics.

Preprocessing checks, and updates:
1. Check if there are any flows where it is apparent that the connection was not established properly (3-way handshake / missing SYN and ACK flags).\
Here I found out, that there are no such flows in the dataset.

2. Check if there are any flows which might not have been finished properly (missing FIN flag).\
Here I found `177 377` such flows. Accordingly, I decided to remove them, as they might be missing relevant information about the flow.

3. While exploring the unique values of the `TLS_SNI` attribute, I found out that there are values, which could be "aggregated".\
For example, consider `TLS_SNI` values containing `safeframe.googlesyndication.com`, which additionally contain an ID/hash as sub-subdomain.\
Rewriting such values to `safeframe.googlesyndication.com` will significantly reduce the number of `TLS_SNI` unique values, which might improve results\
of clustering algorithms, which will be employed in the second part of the project.

The steps above reduced the number of unique `TLS_SNI` values from `19 910` to `7 881`.

Engineered network flow features, which might be interesting for further analysis:
- `float FLOW_DURATION` - flow duration in seconds
- `float AVG_PACKET_SIZE` - average outgoing packet size
- `float AVG_PACKET_SIZE_REV` - average incoming packet size
- `int TOTAL_PACKETS` - total number of packets
- `int TOTAL_BYTES` - total number of bytes
- `float TIME_BETWEEN_PACKETS` - average time between packets in seconds
- `float TRAFFIC_SPEED` - overall traffic speed in bytes per second
- `float TRAFFIC_RATIO` - ratio of sent/received packets (upload/download), where <1 download / >1 upload is major

Basic statistical data of the attributes after the preprocessing is displayed at the end of this subsection.

In [ ]:
# Check if flows start with 3-way handshake as it should in TCP. (SYN, SYN+ACK, ACK)
# https://www.rfc-editor.org/rfc/rfc9293.html#name-establishing-a-connection
# Count the number of flows which could not have been established properly - missing SYN and ACK flags.
df[df["uint8 TCP_FLAGS"] & 0b0001_0010 == 0].shape[0]

In [ ]:
# Count flows which were not finished properly.
# # https://www.ietf.org/rfc/rfc9293.html#name-header-format
df[df["uint8 TCP_FLAGS"] & 0b0000_0001 == 0].shape[0]

In [ ]:
# Remove flows which were not finished properly.
df = df[df["uint8 TCP_FLAGS"] & 0b0000_0001 == 1]

# Size of the reduced dataset.
df.shape

In [ ]:
# Example of TLS_SNI values which contain some ID/hash, and significantly increase the number of unique values, though they are the same service.
df[df["string TLS_SNI"].str.contains("safeframe.googlesyndication.com")].head()

In [ ]:
# Rewrite some TLS_SNI values which contain some ID/hash to reduce the number of unique values.
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.safeframe.googlesyndication.com", value="safeframe.googlesyndication.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.googlevideo.com", value="googlevideo.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.ingest.sentry.io", value="ingest.sentry.io", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.c.2mdn.net", value="c.2mdn.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.fls.doubleclick.net", value="fls.doubleclick.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.metric.gstatic.com", value="metric.gstatic.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.c.drive.google.com", value="c.drive.google.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.cloudfront.net", value="cloudfront.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.ampproject.net", value="ampproject.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.eshop-rychle.cz", value="eshop-rychle.cz", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.myshopify.com", value="myshopify.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.webpkgcache.com", value="webpkgcache.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace="collector.*.px-cloud.net", value="collector.px-cloud.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace="collector.*.perimeterx.net", value="collector.perimeterx.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.prmutv.co", value="prmutv.co", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.tapecontent.net", value="tapecontent.net", regex=True)

In [ ]:
# New number of unique TLS_SNI values.
df["string TLS_SNI"].nunique()

In [ ]:
# Engineering new features.
if "time TIME_FIRST" in df.columns and "time TIME_LAST" in df.columns:  # Workaround for re-running the cell.
    # Flow duration in ns.
    df["float FLOW_DURATION"] = (df["time TIME_LAST"] - df["time TIME_FIRST"]).dt.total_seconds()
    # Mean packet size.
    df["float AVG_PACKET_SIZE"] = df["uint64 BYTES"] / df["uint32 PACKETS"]
    # Mean packet size reverse.
    df["float AVG_PACKET_SIZE_REV"] = df["uint64 BYTES_REV"] / df["uint32 PACKETS_REV"]
    # Number of packets.
    df["int TOTAL_PACKETS"] = df["uint32 PACKETS"] + df["uint32 PACKETS_REV"]
    # Number of bytes.
    df["int TOTAL_BYTES"] = df["uint64 BYTES"] + df["uint64 BYTES_REV"]
    # Time between packets.
    df["float TIME_BETWEEN_PACKETS"] = df["float FLOW_DURATION"] / df["int TOTAL_PACKETS"]
    # Download/upload speed. Convert time to seconds.
    df["float TRAFFIC_SPEED"] = df["int TOTAL_BYTES"] / df["float FLOW_DURATION"]
    # Download/Upload ratio (<1 download / >1 upload is major).
    df["float TRAFFIC_RATIO"] = df["uint32 PACKETS"] / df["uint32 PACKETS_REV"]

    # Drop columns, which might not be needed anymore.
    df = df.drop(columns=["time TIME_FIRST", "time TIME_LAST"])

df.describe()

## Visualization

This section presents information about dataset attributes. Starting with the names of services which communicate the most and least frequently.
That is followed by the names of services which transfer the most data. The last part of the section shows violin plots of numerical attributes.
Values in the violin plots are log10 transformed to better visualize the data distribution, because of the wide range of values.

In [ ]:
# Show names of services which communicate the most frequently.
service_frequency = df["string TLS_SNI"].value_counts()
service_frequency.head(50)

In [ ]:
# Show some of the least frequent services.
service_frequency.tail(50)

In [ ]:
# Show names of services which transfer the most data.
service_bytes = df[["string TLS_SNI", "int TOTAL_BYTES"]].groupby("string TLS_SNI").sum().sort_values(by="int TOTAL_BYTES", ascending=False)
service_bytes.head(50)

In [ ]:
# Show some of the services which transfer the least data.
service_bytes.tail(50)

In [ ]:
# List of numerical columns in your DataFrame
num_cols = [
    "uint32 PACKETS", "uint32 PACKETS_REV",
    "uint64 BYTES", "uint64 BYTES_REV",
    "float AVG_PACKET_SIZE", "float AVG_PACKET_SIZE_REV",
    "float FLOW_DURATION", "float TIME_BETWEEN_PACKETS",
    "int TOTAL_PACKETS", "int TOTAL_BYTES",
    "float TRAFFIC_SPEED", "float TRAFFIC_RATIO"
]
titles = [
    "Number of outgoing packets",
    "Number of incoming packets",
    "Number of sent bytes (B)",
    "Number of received bytes (B)",
    "Average outgoing packet size (B)",
    "Average incoming packet size (B)",
    "Flow duration (s)",
    "Average time between packets (s)",
    "Total number of transmitted packets in flow",
    "Total number of transmitted bytes in flow",
    "Traffic speed (bytes/s)",
    "Traffic ratio (upload/download)",
]
# Create a new figure and axes with a layout of 3 rows and 3 columns
fig, axs = plt.subplots(6, 2, figsize=(20, 30))

# Flatten the axes array
axs = axs.flatten()

# For each numerical column, create a violin plot on a separate subplot
for i, col in enumerate(num_cols):
    sns.violinplot(x=np.log10(df[col]), ax=axs[i])
    axs[i].set_title(titles[i], fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
# Count usage of TCP flags in the dataset.
tcp_flags_dict = {
    "Flag": [
        "CWR",
        "ECE",
        "URG",
        "ACK",
        "PSH",
        "RST",
        "SYN",
        "FIN",
    ],
    "Count": [
        df[df["uint8 TCP_FLAGS"] & 0b1000_0000 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0100_0000 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0010_0000 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0001_0000 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0000_1000 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0000_0100 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0000_0010 != 0].shape[0],
        df[df["uint8 TCP_FLAGS"] & 0b0000_0001 != 0].shape[0],
    ],
}
tcp_flags = pd.DataFrame(tcp_flags_dict)
tcp_flags.plot.bar(x="Flag", y="Count", rot=0)
plt.title("TCP flags presence in the network flows")
plt.show()